# 323. Number of Connected Components in an Undirected Graph
**Difficulty:** 🟡 Medium (Premium) · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/number-of-connected-components-in-an-undirected-graph/

## 💡 Concepts

**Core concept(s):** Count separate groups with **Union-Find** or by counting **traversal starts**.

**Why it applies here:** A connected component is a maximal blob of nodes joined by edges. Either union every edge and count the distinct groups left, or DFS/BFS from each unvisited node — each fresh start is one new component.

**Key intuition:** Start with n lonely nodes; every edge that joins two different groups reduces the group count by one.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is Union-Find (Disjoint Set)?
**Union-Find** tracks which items are in the same group. `find(x)` returns x's group leader; `union(a,b)` merges two groups. With path-compression it's almost **O(1)** per call.
- **Great for:** counting connected pieces, detecting cycles in undirected graphs.
- **In Python:** a `parent` array where each node points toward its group's leader.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

---

**Prerequisite knowledge:**
- Union-Find, or traversal with a visited set.

## 📝 Problem

Given `n` nodes and undirected `edges`, count the connected components.

**Example**
```
5, [[0,1],[1,2],[3,4]] -> 2
5, [[0,1],[1,2],[2,3],[3,4]] -> 1
```

> Two approaches, both `O(n + edges)`: Union-Find and traversal.

### Approach 1 — Union-Find

**Idea:** Start with `n` groups. Each edge that joins two different groups reduces the count by one.

**Time:** `O((n+E)·α) ≈ O(n+E)`. **Space:** `O(n)`.

In [ ]:
def count_components_uf(n, edges):
    parent = list(range(n))                # each node starts alone
    count = n                              # start with n separate groups
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:                       # this edge joins two different groups
            parent[ra] = rb                # merge them...
            count -= 1                      # ...so one fewer group remains
    return count

### Approach 2 — Traversal, Count Starts

**Idea:** DFS/BFS from each unvisited node; every fresh start marks a new component.

**Time:** `O(n + E)`. **Space:** `O(n + E)`.

In [ ]:
from collections import defaultdict

def count_components_dfs(n, edges):
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b); graph[b].append(a)
    seen = set(); count = 0
    for i in range(n):
        if i not in seen:                  # a node we haven't reached -> a NEW component
            count += 1
            stack = [i]                    # explore this whole component
            while stack:
                node = stack.pop()
                if node in seen:
                    continue
                seen.add(node)
                for nb in graph[node]:
                    if nb not in seen:
                        stack.append(nb)
    return count

In [ ]:
# Correctness check
tests = [(5,[[0,1],[1,2],[3,4]],2), (5,[[0,1],[1,2],[2,3],[3,4]],1), (4,[],4)]
for n, edges, exp in tests:
    a, b = count_components_uf(n, edges), count_components_dfs(n, edges)
    print(f"n={n}, edges={edges} -> uf={a}, dfs={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    edges = [[i, i + 1] for i in range(0, n - 1, 2)]   # pairs -> ~n/2 components
    return (n, edges)
solutions = {
    "union-find O(n+E)": count_components_uf,
    "traversal  O(n+E)": count_components_dfs,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Union-Find counts groups:** start at n, subtract one per merging edge.
- **Traversal counts starts:** each unvisited node begins a new component.
- **Signal:** "connected components / groups / friend circles / provinces".
- **Related problems:** Graph Valid Tree, Number of Islands, Redundant Connection, Accounts Merge.
- **Common pitfalls:** (1) counting an edge that joins already-connected nodes; (2) missing isolated nodes with no edges.